In [ ]:
%%sql


# Criticism Data Collection Runbook

This notebook helps collaborators run data collection without editing package code. It performs network collection when enabled, can run for many hours on multi-year GDELT DOC collection, and can be interrupted safely because successful DOC day payloads are checkpointed.

Raw outputs are never overwritten. Existing daily DOC checkpoints are reused. This notebook is a runbook, not core pipeline logic; the authoritative implementation remains in `src/mci/` and `scripts/`.

## Smoke-Test Parameters

Start here. A one-day run verifies credentials, network access, deterministic paths, checkpointing, and previews without launching a long collection.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd


def find_project_root(start: Path) -> Path:
    candidates = []
    project_root_override = globals().get("PROJECT_ROOT_OVERRIDE")
    if project_root_override:
        candidates.append(Path(project_root_override).expanduser())

    env_project_root = os.environ.get("MCI_PROJECT_ROOT")
    if env_project_root:
        candidates.append(Path(env_project_root).expanduser())

    candidates.extend([start, *start.parents])
    candidates.extend([
        start / "market-criticism-index",
        Path.home() / "Desktop" / "market-criticism-index",
        Path.home() / "market-criticism-index",
    ])

    seen_candidates = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen_candidates:
            continue
        seen_candidates.add(candidate)
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "mci").exists():
            return candidate
    raise RuntimeError("Could not find the project root. Start Jupyter from the repository checkout, set PROJECT_ROOT_OVERRIDE in the Parameters cell, or set MCI_PROJECT_ROOT.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mci.gdelt import (
    GdeltClient,
    GdeltClientError,
    GdeltLongRunSpec,
    GdeltQueryType,
    GkgBulkSpec,
    collect_gdelt_longrun,
    collect_gkg_bulk_extract,
    dry_run_gdelt_longrun,
    gdelt_longrun_status,
)
from mci.market_data import     MarketDataSpec, collect_market_data

PROJECT_ROOT

WindowsPath('D:/Work/market-criticism-index')

In [2]:
from datetime import date
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
START_DATE = date(2022, 1, 3)
END_DATE = date(2022, 1, 3)
MAX_RECORDS = 25
RUN_GDELT = True
RUN_MARKET_DATA = False
RESUME = True
OVERWRITE_INTERIM = True
SOURCE_LANGUAGE = "english"

REQUEST_PAUSE_SECONDS = 10
MAX_RETRIES = 10
BACKOFF_SECONDS = 30
MAX_BACKOFF_SECONDS = 1800
JITTER_SECONDS = 10

MARKET_MAX_RETRIES = 5
MARKET_BACKOFF_SECONDS = 60.0
MARKET_MAX_BACKOFF_SECONDS = 900.0
MARKET_REQUEST_PAUSE_SECONDS = 5.0

## Full MVP Date Range

After the smoke test succeeds, uncomment this cell and rerun the dry-run/status cells before starting collection. A full two-query DOC run may take many hours.

In [3]:
START_DATE = date(2022,1,1)
END_DATE = date(2026,5,31)
MAX_RECORDS = 250
RUN_GDELT = True
RUN_MARKET_DATA = False
RESUME = True
OVERWRITE_INTERIM = False
SOURCE_LANGUAGE = "english"

## Import And Setup

In [4]:
def make_doc_spec(query_types):
    return GdeltLongRunSpec(
        start_date=START_DATE,
        end_date=END_DATE,
        query_types=tuple(query_types),
        max_records=MAX_RECORDS,
        resume=RESUME,
        overwrite_interim=OVERWRITE_INTERIM,
        source_language=SOURCE_LANGUAGE,
        request_pause_seconds=REQUEST_PAUSE_SECONDS,
    )


def make_client():
    return GdeltClient(
        max_retries=MAX_RETRIES,
        backoff_seconds=BACKOFF_SECONDS,
        max_backoff_seconds=MAX_BACKOFF_SECONDS,
        jitter_seconds=JITTER_SECONDS,
        request_pause_seconds=REQUEST_PAUSE_SECONDS,
    )


def print_progress(event):
    phase = event.get("phase")
    query_type = event.get("query_type", "")
    day = event.get("date", "")
    if phase == "retry":
        status = event.get("status_code") or event.get("error")
        print(f"{day} {query_type} retry {event.get('attempt')} after {status}; sleeping {event.get('sleep_seconds')}s")
    elif phase == "skip":
        print(f"{day} {query_type} skipped existing checkpoint")
    elif phase == "checkpoint":
        print(f"{day} {query_type} checkpoint saved")
    elif phase == "fatal_failure":
        print(f"{day} {query_type} fatal failure: {event}")
    elif phase == "aggregate":
        print(f"{query_type} aggregate saved: {event.get('article_count')} articles")
    elif phase and phase.startswith("gkg"):
        print(event)


def print_doc_result(result):
    for query_type, raw_path in result.raw_paths.items():
        print(f"{query_type}")
        print(f"  Raw JSON: {raw_path}")
        print(f"  Interim CSV: {result.interim_paths[query_type]}")
        print(f"  Articles: {result.article_counts[query_type]}")

## Dry Run And Checkpoint Status

This does not call the network. It shows request counts, existing checkpoints, missing checkpoints, and estimated minimum duration from the configured polite pause.

In [5]:
doc_spec = make_doc_spec((GdeltQueryType.ALL_MARKET, GdeltQueryType.CANDIDATE_CRITICISM))
dry_run_gdelt_longrun(doc_spec)

,query_type,start_date,end_date,request_count,existing_checkpoints,missing_checkpoints,estimated_minimum_duration_seconds
0,all_us_market,2022-01-01,2026-05-31,1612,1,1611,16110
1,candidate_criticism,2022-01-01,2026-05-31,1612,0,1612,16120


In [6]:
status = gdelt_longrun_status(doc_spec)
coverage = status.groupby("query_type")["checkpoint_exists"].agg(["sum", "count"])
coverage["missing"] = coverage["count"] - coverage["sum"]
coverage

,sum,count,missing
query_type,,,
all_us_market,1,1612,1611
candidate_criticism,0,1612,1612


## Collect All US-Market DOC Headlines

Rerunning this cell resumes from daily checkpoints.

In [7]:
all_market_result = None
if RUN_GDELT:
    all_market_spec = make_doc_spec((GdeltQueryType.ALL_MARKET,))
    try:
        all_market_result = collect_gdelt_longrun(all_market_spec, client=make_client(), progress_callback=print_progress)
        print_doc_result(all_market_result)
    except FileExistsError as exc:
        print("Outputs already exist. Use existing files, change dates, or keep RESUME=True for daily checkpoints. Raw aggregate outputs are never overwritten.")
        print(exc)
    except GdeltClientError as exc:
        print(exc)
        if exc.resume_guidance:
            print("\nResume snippet:\n" + exc.resume_guidance)
else:
    print("Skipped all-market DOC collection because RUN_GDELT is False.")

2022-01-01 all_us_market retry 1 after 429; sleeping 36.70270873544603s
2022-01-01 all_us_market retry 2 after 429; sleeping 69.75785905933002s
2022-01-01 all_us_market checkpoint saved
2022-01-02 all_us_market retry 1 after 429; sleeping 38.70266453241324s
2022-01-02 all_us_market retry 2 after 429; sleeping 60.74966207525395s
2022-01-02 all_us_market retry 3 after 429; sleeping 127.95940186828268s
2022-01-02 all_us_market retry 4 after 429; sleeping 242.05666680825473s
2022-01-02 all_us_market retry 5 after 429; sleeping 483.0219185838974s
2022-01-02 all_us_market retry 6 after 429; sleeping 964.1457961721906s
2022-01-02 all_us_market retry 7 after 429; sleeping 1800s
2022-01-02 all_us_market retry 8 after 429; sleeping 1800s
2022-01-02 all_us_market retry 9 after 429; sleeping 1800s
2022-01-02 all_us_market checkpoint saved
2022-01-03 all_us_market skipped existing checkpoint
2022-01-04 all_us_market retry 1 after 429; sleeping 33.79806745264465s
2022-01-04 all_us_market retry 2 aft

## Collect Candidate-Criticism DOC Headlines

Rerunning this cell resumes from daily checkpoints.

In [8]:
candidate_result = None
if RUN_GDELT:
    candidate_spec = make_doc_spec((GdeltQueryType.CANDIDATE_CRITICISM,))
    try:
        candidate_result = collect_gdelt_longrun(candidate_spec, client=make_client(), progress_callback=print_progress)
        print_doc_result(candidate_result)
    except FileExistsError as exc:
        print("Outputs already exist. Use existing files, change dates, or keep RESUME=True for daily checkpoints. Raw aggregate outputs are never overwritten.")
        print(exc)
    except GdeltClientError as exc:
        print(exc)
        if exc.resume_guidance:
            print("\nResume snippet:\n" + exc.resume_guidance)
else:
    print("Skipped candidate-criticism DOC collection because RUN_GDELT is False.")

2022-01-01 candidate_criticism fatal failure: {'query_type': 'candidate_criticism', 'date': '2022-01-01', 'phase': 'fatal_failure', 'error': 'invalid json'}
GDELT returned a response that was not valid JSON.
Completed checkpoint days for candidate_criticism: 0.
Next missing day: 2022-01-01.
Resume guidance:
spec = GdeltLongRunSpec(
    start_date=date(2022, 1, 1),
    end_date=date(2026, 5, 31),
    query_types=(GdeltQueryType.CANDIDATE_CRITICISM,),
    max_records=250,
    source_language='english',
    raw_dir=Path('D:\\Work\\market-criticism-index\\data\\raw\\gdelt'),
    interim_dir=Path('D:\\Work\\market-criticism-index\\data\\interim'),
    resume=True,
    overwrite_interim=False,
)
collect_gdelt_longrun(spec, progress_callback=print_progress)

Resume snippet:
spec = GdeltLongRunSpec(
    start_date=date(2022, 1, 1),
    end_date=date(2026, 5, 31),
    query_types=(GdeltQueryType.CANDIDATE_CRITICISM,),
    max_records=250,
    source_language='english',
    raw_dir=Path('D:\\Wor

## Resume Status After Interruption

Run this after a kernel restart or interrupted collection. The first missing date is where the next resumed run starts fetching.

In [9]:
status = gdelt_longrun_status(doc_spec)
missing = status[~status["checkpoint_exists"]]
print(f"checkpoint coverage: {len(status) - len(missing)}/{len(status)} days complete")
if missing.empty:
    print("All requested DOC checkpoints are complete.")
else:
    print(f"Next missing day: {missing.iloc[0]['date']} ({missing.iloc[0]['query_type']})")
    display(missing.head())

checkpoint coverage: 19/3224 days complete
Next missing day: 2022-01-20 (all_us_market)


,query_type,date,checkpoint_path,checkpoint_exists,aggregate_raw_path,interim_path
19,all_us_market,2022-01-20,D:\Work\market-criticism-index\data\raw\gdelt\...,False,D:\Work\market-criticism-index\data\raw\gdelt\...,D:\Work\market-criticism-index\data\interim\gd...
20,all_us_market,2022-01-21,D:\Work\market-criticism-index\data\raw\gdelt\...,False,D:\Work\market-criticism-index\data\raw\gdelt\...,D:\Work\market-criticism-index\data\interim\gd...
21,all_us_market,2022-01-22,D:\Work\market-criticism-index\data\raw\gdelt\...,False,D:\Work\market-criticism-index\data\raw\gdelt\...,D:\Work\market-criticism-index\data\interim\gd...
22,all_us_market,2022-01-23,D:\Work\market-criticism-index\data\raw\gdelt\...,False,D:\Work\market-criticism-index\data\raw\gdelt\...,D:\Work\market-criticism-index\data\interim\gd...
23,all_us_market,2022-01-24,D:\Work\market-criticism-index\data\raw\gdelt\...,False,D:\Work\market-criticism-index\data\raw\gdelt\...,D:\Work\market-criticism-index\data\interim\gd...


## Collect Market Data

This fetches SPY, QQQ, RSP, and VIX through `collect_market_data`. Raw market data is cached under `data/raw/market/` and is never overwritten.

In [10]:
market_data_path = None
if RUN_MARKET_DATA:
    try:
        market_spec = MarketDataSpec(
            start_date=START_DATE,
            end_date=END_DATE,
            max_retries=MARKET_MAX_RETRIES,
            backoff_seconds=MARKET_BACKOFF_SECONDS,
            max_backoff_seconds=MARKET_MAX_BACKOFF_SECONDS,
            request_pause_seconds=MARKET_REQUEST_PAUSE_SECONDS,
        )
        market_data_path = collect_market_data(market_spec)
        print(f"Saved raw market CSV: {market_data_path}")
    except FileExistsError as exc:
        print("Raw market-data output already exists. Use the existing file or change START_DATE/END_DATE. Raw data is never overwritten.")
        print(exc)
    except (RuntimeError, ValueError) as exc:
        print("Market data collection did not complete.")
        print(exc)
        print("Recovery: wait before retrying, change START_DATE/END_DATE, or reuse an existing raw market CSV if present.")
else:
    print("Skipped market-data collection because RUN_MARKET_DATA is False.")

Skipped market-data collection because RUN_MARKET_DATA is False.


## Sanity Checks

In [11]:
def preview_csv(path: Path, rows: int = 5):
    data = pd.read_csv(path)
    print(f"{path}")
    print(f"Rows: {len(data)}")
    if "date" in data.columns:
        print(f"Date range: {data['date'].min()} to {data['date'].max()}")
    elif "seendate" in data.columns and not data.empty:
        print(f"First seendate: {data['seendate'].iloc[0]}")
    display(data.head(rows))


for result in (all_market_result, candidate_result):
    if result is None:
        continue
    for path in result.interim_paths.values():
        preview_csv(path)

if market_data_path is not None:
    preview_csv(market_data_path)

## Optional Historical GKG Fallback

This is an audit/candidate-extraction fallback, not equivalent to DOC ArticleList headline rows. It streams GKG zip archives, writes filtered extracts plus a manifest, and does not retain full zip archives unless `GKG_KEEP_ARCHIVES=True`.

In [12]:
RUN_GKG_BULK = False
GKG_KEEP_ARCHIVES = False
GKG_MAX_ARCHIVES = 2  # keep this low for a smoke test

if RUN_GKG_BULK:
    gkg_spec = GkgBulkSpec(
        start_date=START_DATE,
        end_date=END_DATE,
        filter_terms=("stock market", "Wall Street", "S&P 500", "Nasdaq", "bubble", "overvalued"),
        keep_archives=GKG_KEEP_ARCHIVES,
        max_archives=GKG_MAX_ARCHIVES,
    )
    try:
        gkg_result = collect_gkg_bulk_extract(gkg_spec, progress_callback=print_progress)
        print(f"Filtered GKG extract: {gkg_result.extract_path}")
        print(f"Manifest: {gkg_result.manifest_path}")
        print(f"Archives: {gkg_result.archive_count}; matched rows: {gkg_result.matched_row_count}")
    except FileExistsError as exc:
        print("Filtered GKG outputs already exist. Use existing files or change dates. Raw fallback extracts are not overwritten.")
        print(exc)
else:
    print("Skipped optional GKG fallback because RUN_GKG_BULK is False.")

Skipped optional GKG fallback because RUN_GKG_BULK is False.


## Next Steps

Use scripts and package functions for the reproducible pipeline. Do not move processing logic into this notebook.

```bash
python scripts/build_annotation_sample.py --candidate-csv data/interim/gdelt_candidate_criticism_20220101_20260531.csv --all-market-csv data/interim/gdelt_all_us_market_20220101_20260531.csv --seed 1
python scripts/build_mci_daily.py --market-csv data/interim/gdelt_all_us_market_20220101_20260531.csv --criticism-csv data/interim/gdelt_candidate_criticism_20220101_20260531.csv --overwrite
python scripts/build_panel_daily.py --prices-csv data/raw/market/market_prices_spy_qqq_rsp_vix_20220101_20260531.csv --overwrite
python scripts/run_mvp_analysis.py --overwrite
```

Before MCI construction, run the cleaning and trading-day alignment steps appropriate for the collected headline files.